In [1]:
from abc import ABC, abstractmethod

class Personaje(ABC):
    def __init__(self, nombre, nivel, vida_max):
        self.nombre = nombre
        self.nivel = nivel
        self.puntos_vida_max = vida_max
        self.puntos_vida = vida_max

    def recibir_dano(self, dano):
        self.puntos_vida = max(0, self.puntos_vida - dano)
        print(f"{self.nombre} recibe {dano} de daño. HP: {self.puntos_vida}/{self.puntos_vida_max}")

    def esta_vivo(self):
        return self.puntos_vida > 0

    @abstractmethod
    def atacar(self, objetivo): pass

    @abstractmethod
    def get_tipo_personaje(self): pass

    def __str__(self):
        return f"[{self.get_tipo_personaje():<7}] {self.nombre:<7} Nv.{self.nivel} │ HP: {self.puntos_vida}/{self.puntos_vida_max}"

In [2]:
class Habilidoso(ABC):
    @abstractmethod
    def usar_habilidad_especial(self, objetivo): pass
    @abstractmethod
    def get_costo_habilidad(self): pass
    @abstractmethod
    def get_nombre_habilidad(self): pass

class Equipable(ABC):
    @abstractmethod
    def equipar_item(self, item): pass
    @abstractmethod
    def get_item_equipado(self): pass

class Sanador(ABC):
    @abstractmethod
    def sanar(self, objetivo): pass
    @abstractmethod
    def get_potencia_sanacion(self): pass

In [3]:
class Guerrero(Personaje, Habilidoso, Equipable):
    def __init__(self, nombre, nivel):
        super().__init__(nombre, nivel, 100 + nivel * 10)
        self.fuerza = 15 + nivel * 3
        self.defensa = 10 + nivel * 2
        self.item_equipado = "Sin equipo"
        self.costo_habilidad = 30

    def atacar(self, objetivo):
        print(f"{self.nombre} golpea a {objetivo.nombre} con fuerza {self.fuerza}!")
        objetivo.recibir_dano(self.fuerza)

    def get_tipo_personaje(self): return "Guerrero"

    def usar_habilidad_especial(self, objetivo):
        print(f"{self.nombre} usa ¡Golpe Devastador! sobre {objetivo.nombre}")
        objetivo.recibir_dano(50)

    def get_costo_habilidad(self): return self.costo_habilidad
    def get_nombre_habilidad(self): return "Golpe Devastador"

    def equipar_item(self, item):
        self.item_equipado = item
        print(f"{self.nombre} equipó: {item}")

    def get_item_equipado(self): return self.item_equipado


class Mago(Personaje, Habilidoso, Sanador):
    def __init__(self, nombre, nivel):
        super().__init__(nombre, nivel, 60 + nivel * 5)
        self.mana_max = 80 + nivel * 10
        self.mana = self.mana_max

    def atacar(self, objetivo):
        if self.mana >= 20:
            dano = 25 + self.nivel * 5
            self.mana -= 20
            print(f"{self.nombre} lanza un hechizo sobre {objetivo.nombre}!")
            objetivo.recibir_dano(dano)
        else:
            print(f"{self.nombre} no tiene maná suficiente.")

    def get_tipo_personaje(self): return "Mago"

    def usar_habilidad_especial(self, objetivo):
        if self.mana >= 20:
            self.mana -= 20
            print(f"{self.nombre} lanza ¡Bola de Fuego! sobre {objetivo.nombre}")
            objetivo.recibir_dano(40)
        else:
            print(f"{self.nombre} sin maná para Bola de Fuego.")

    def get_costo_habilidad(self): return 20
    def get_nombre_habilidad(self): return "Bola de Fuego"

    def sanar(self, objetivo):
        curado = min(25, objetivo.puntos_vida_max - objetivo.puntos_vida)
        objetivo.puntos_vida += curado
        print(f"{self.nombre} sana a {objetivo.nombre} por {curado} HP.")

    def get_potencia_sanacion(self): return 25


class Arquero(Personaje, Equipable):
    def __init__(self, nombre, nivel):
        super().__init__(nombre, nivel, 75 + nivel * 7)
        self.flechas = 10 + nivel * 2
        self.alcance = 30
        self.item_equipado = "Arco básico"

    def atacar(self, objetivo):
        if self.flechas > 0:
            dano = 12 + self.nivel * 4
            if self.item_equipado != "Arco básico":
                dano += 5
            self.flechas -= 1
            print(f"{self.nombre} dispara a {objetivo.nombre}! (Flechas: {self.flechas})")
            objetivo.recibir_dano(dano)
        else:
            print(f"{self.nombre} no tiene flechas.")

    def get_tipo_personaje(self): return "Arquero"

    def equipar_item(self, item):
        self.item_equipado = item
        print(f"{self.nombre} equipó: {item}")

    def get_item_equipado(self): return self.item_equipado

In [4]:
heroes = [Guerrero('Thorin', 3), Mago('Gandalf', 5), Arquero('Legolas', 4)]
orco = Guerrero('Orco', 1)
turno = 1

while orco.esta_vivo():
    print(f"\n--- Turno {turno} ---")
    for h in heroes:
        if orco.esta_vivo():
            h.atacar(orco)
    turno += 1

print(f"\nBatalla terminada en {turno - 1} turno(s).")
for h in heroes: print(h)
print(orco)


--- Turno 1 ---
Thorin golpea a Orco con fuerza 24!
Orco recibe 24 de daño. HP: 86/110
Gandalf lanza un hechizo sobre Orco!
Orco recibe 50 de daño. HP: 36/110
Legolas dispara a Orco! (Flechas: 17)
Orco recibe 28 de daño. HP: 8/110

--- Turno 2 ---
Thorin golpea a Orco con fuerza 24!
Orco recibe 24 de daño. HP: 0/110

Batalla terminada en 2 turno(s).
[Guerrero] Thorin  Nv.3 │ HP: 130/130
[Mago   ] Gandalf Nv.5 │ HP: 85/85
[Arquero] Legolas Nv.4 │ HP: 103/103
[Guerrero] Orco    Nv.1 │ HP: 0/110
